# Policy Agent — Team Walkthrough

This notebook walks your team through **building, deploying, and testing** the policy agent end-to-end:

1. Environment check (gcloud, ADC, Python)
2. Install dependencies
3. Configure project / region / buckets
4. Create GCS buckets and seed sample policies
5. Local smoke test (no deploy)
6. Deploy to Vertex AI Agent Engine with **Agent Identity**
7. Grant the agent principal `roles/storage.objectViewer` on the policy bucket
8. Remote test against the deployed reasoning engine
9. *(optional)* Register as a Gemini Enterprise add-on agent
10. Teardown (stop billing when you're done)



## 1. Environment check

Verify the prerequisites before you do anything that costs money.

**Common gotcha:** the `gcloud` CLI active account and your Application Default Credentials (ADC) can be *different* accounts. The Python SDK uses ADC, so if they differ you'll see `403 storage.buckets.get` during deploy. The cell below prints both — they should match (or at least both have access to the target project).

In [ ]:
import shutil, subprocess, sys

print('Python :', sys.version.split()[0])
for tool in ('gcloud', 'gsutil'):
    path = shutil.which(tool)
    print(f'{tool:7}: {path or "NOT FOUND — install the Google Cloud SDK"}')

def sh(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except subprocess.CalledProcessError as e:
        return f'ERROR: {e.output.strip()}'

print('\ngcloud active account :', sh('gcloud config get-value account'))
print('gcloud active project :', sh('gcloud config get-value project'))
print('\nADC identity (this is what the Python SDK will use):')
print(sh('gcloud auth application-default print-access-token | head -c 1 > /dev/null && '
        'curl -s -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" '
        'https://openidconnect.googleapis.com/v1/userinfo'))

If ADC is missing or wrong, run this in a terminal (opens a browser), then restart the kernel:
```
gcloud auth application-default login --account=<your-account>
```

## 2. Install dependencies

Installs the package in editable mode so code edits in `policy_agent/` are picked up without reinstalling.

In [ ]:
%pip install -q -e ..

## 3. Configuration

**Edit this cell** — everything below reads from these variables. Keep bucket names globally unique (they share a namespace across all of Google Cloud).

In [ ]:
import os

PROJECT_ID     = os.environ.get('GOOGLE_CLOUD_PROJECT', 'your-project-id')
LOCATION       = os.environ.get('GOOGLE_CLOUD_LOCATION', 'us-central1')
STAGING_BUCKET = f'{PROJECT_ID}-agent-staging'
POLICY_BUCKET  = f'{PROJECT_ID}-policies'
POLICY_PREFIX  = 'policies/'
DISPLAY_NAME   = 'policy-agent'
AGENT_MODEL    = 'gemini-2.5-flash'

# Optional — only used in Step 9.
GEMINI_ENTERPRISE_APP_ID  = os.environ.get('GEMINI_ENTERPRISE_APP_ID', '')
GEMINI_ENTERPRISE_REGION  = os.environ.get('GEMINI_ENTERPRISE_LOCATION', 'global')

# Propagate to the child processes that deploy.py / tools will spawn
os.environ['GOOGLE_CLOUD_PROJECT']  = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION
os.environ['POLICY_BUCKET']         = POLICY_BUCKET
os.environ['POLICY_PREFIX']         = POLICY_PREFIX

for k in ('PROJECT_ID','LOCATION','STAGING_BUCKET','POLICY_BUCKET','POLICY_PREFIX','AGENT_MODEL'):
    print(f'{k:16} = {eval(k)}')
assert PROJECT_ID and PROJECT_ID != 'your-project-id', 'Set PROJECT_ID before proceeding'

## 4. Enable APIs, create buckets, seed sample policies

Idempotent — re-runs no-op if the API is already enabled or the bucket already exists.

In [ ]:
import subprocess

def run(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0 and 'already exists' not in (r.stderr or '').lower():
        print('STDERR:', r.stderr.strip())
    return r

run(f'gcloud services enable aiplatform.googleapis.com storage.googleapis.com discoveryengine.googleapis.com --project={PROJECT_ID}')
run(f'gcloud storage buckets create gs://{STAGING_BUCKET} --project={PROJECT_ID} --location={LOCATION} --uniform-bucket-level-access')
run(f'gcloud storage buckets create gs://{POLICY_BUCKET}  --project={PROJECT_ID} --location={LOCATION} --uniform-bucket-level-access')
run(f'gcloud storage cp ../sample_policies/*.md gs://{POLICY_BUCKET}/{POLICY_PREFIX} --project={PROJECT_ID}')
run(f'gcloud storage ls gs://{POLICY_BUCKET}/{POLICY_PREFIX} --project={PROJECT_ID}')

## 5. Local smoke test

Runs the agent against your personal ADC using ADK's `InMemoryRunner`. This catches tool bugs before a 10-minute deploy cycle. Agent Identity itself is only minted on Agent Engine, so what's being tested here is the tool code path.

In [ ]:
import asyncio, sys
sys.path.insert(0, '..')
from google.adk.runners import InMemoryRunner
from google.genai import types
from policy_agent.agent import root_agent

async def ask_local(prompt: str) -> str:
    runner = InMemoryRunner(agent=root_agent, app_name='policy-agent-nb')
    session = await runner.session_service.create_session(app_name='policy-agent-nb', user_id='nb-user')
    reply = []
    async for event in runner.run_async(
        user_id='nb-user',
        session_id=session.id,
        new_message=types.Content(role='user', parts=[types.Part(text=prompt)]),
    ):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    reply.append(p.text)
    return ''.join(reply)

answer = await ask_local('What does our data retention policy say about email?')
print(answer)

In [ ]:
# Try a couple more — useful for demoing to the team.
print(await ask_local('Am I allowed to install unapproved software on my work laptop?'))
print('---')
print(await ask_local('Is there a stipend for home office equipment?'))

### 5a. Error-path smoke test (optional)

Proves the tools return a clean structured error instead of raising, when the bucket doesn't exist.

In [ ]:
from policy_agent.gcs_tools import list_policies
list_policies(bucket='definitely-not-a-real-bucket-xyz-12345')

## 6. Deploy to Vertex AI Agent Engine with Agent Identity

**This step creates billable resources** — ~5–15 minutes while the container builds.

The key line is `identity_type=types.IdentityType.AGENT_IDENTITY`. That makes Agent Engine mint a per-agent SPIFFE principal you can grant IAM to directly.

In [ ]:
import vertexai
from vertexai import types
from vertexai.agent_engines import AdkApp
from policy_agent.agent import root_agent

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=dict(api_version='v1beta1'),
)

app = AdkApp(agent=root_agent)

print('Deploying... (this takes ~5-15 minutes)')
remote_app = client.agent_engines.create(
    agent=app,
    config={
        'display_name': DISPLAY_NAME,
        'identity_type': types.IdentityType.AGENT_IDENTITY,
        'extra_packages': ['../policy_agent'],
        'requirements': [
            'google-cloud-aiplatform[adk,agent_engines]',
            'google-cloud-storage>=2.18.0',
            'google-cloud-secret-manager>=2.20.0',
            'pydantic>=2.7.0',
            'cloudpickle>=3.0.0',
        ],
        'staging_bucket': f'gs://{STAGING_BUCKET}',
        # GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION are reserved — don't set them here
        'env_vars': {
            'POLICY_BUCKET': POLICY_BUCKET,
            'POLICY_PREFIX': POLICY_PREFIX,
        },
    },
)

REASONING_ENGINE = remote_app.api_resource.name
print(f'\n✅ Deployed: {REASONING_ENGINE}')

## 7. Grant the agent principal access to GCS

Fetches the reasoning engine's `effectiveIdentity` (the per-agent SPIFFE principal) and binds `roles/storage.objectViewer` on the policy bucket to it.

In [ ]:
import json, subprocess

token = subprocess.check_output('gcloud auth application-default print-access-token', shell=True, text=True).strip()
resp = subprocess.check_output(
    f'curl -s -H "Authorization: Bearer {token}" '
    f'https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}',
    shell=True, text=True,
)
engine = json.loads(resp)
effective_identity = engine['spec']['effectiveIdentity']
AGENT_PRINCIPAL = f'principal://{effective_identity}'
print('Agent principal:', AGENT_PRINCIPAL)

run(
    f'gcloud storage buckets add-iam-policy-binding gs://{POLICY_BUCKET} '
    f'--member="{AGENT_PRINCIPAL}" '
    f'--role="roles/storage.objectViewer" '
    f'--project={PROJECT_ID}'
)

## 8. Remote test

Ask the *deployed* agent — this is what your team / Gemini Enterprise users will hit. The agent authenticates to GCS via its Agent Identity principal (not your ADC).

In [ ]:
remote = client.agent_engines.get(name=REASONING_ENGINE)

async def ask_remote(prompt: str) -> str:
    session = await remote.async_create_session(user_id='nb-user')
    out = []
    async for event in remote.async_stream_query(
        user_id='nb-user', session_id=session['id'], message=prompt,
    ):
        parts = (event.get('content') or {}).get('parts', [])
        for p in parts:
            if 'text' in p:
                out.append(p['text'])
    return ''.join(out)

print(await ask_remote('What does the acceptable use policy say about personal use of company systems?'))

In [ ]:
# Demo error path — temporarily point at a bucket the agent can't read, and verify
# the agent surfaces a clean "I don't have access" message instead of a traceback.
# (Restart a session to clear any cached client.)
print(await ask_remote('Look at bucket some-random-bucket-name-nobody-owns prefix policies/ and list our policies.'))

## 9. *(Optional)* Register as a Gemini Enterprise add-on agent

Requires a Gemini Enterprise app already set up in this project. Skip this section if you're only exposing the agent through the notebook / direct Agent Engine calls.

In [ ]:
import json, urllib.request, subprocess

assert GEMINI_ENTERPRISE_APP_ID, 'Set GEMINI_ENTERPRISE_APP_ID in Step 3 before running this cell'

host = 'discoveryengine.googleapis.com' if GEMINI_ENTERPRISE_REGION == 'global' else f'{GEMINI_ENTERPRISE_REGION}-discoveryengine.googleapis.com'
url = (
    f'https://{host}/v1alpha/projects/{PROJECT_ID}/locations/global/'
    f'collections/default_collection/engines/{GEMINI_ENTERPRISE_APP_ID}/'
    'assistants/default_assistant/agents'
)
payload = {
    'displayName': 'Policy Assistant',
    'description': (
        "Answers questions about internal organizational policies by retrieving "
        "documents from Google Cloud Storage. Invoke for questions about company "
        "rules, HR policies, security policies, acceptable use, or compliance."
    ),
    'adkAgentDefinition': {
        'provisionedReasoningEngine': {'reasoningEngine': REASONING_ENGINE},
    },
}
token = subprocess.check_output('gcloud auth print-access-token', shell=True, text=True).strip()
req = urllib.request.Request(
    url, data=json.dumps(payload).encode(), method='POST',
    headers={
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
        'X-Goog-User-Project': PROJECT_ID,
    },
)
with urllib.request.urlopen(req) as resp:
    print('HTTP', resp.status)
    print(resp.read().decode())

## 10. Teardown (stop ongoing billing)

Deletes the reasoning engine. The buckets stay (empty buckets cost ~nothing) — the stale IAM binding on the policy bucket becomes a no-op once the principal is gone, but you can remove it if you like.

In [ ]:
import subprocess
token = subprocess.check_output('gcloud auth application-default print-access-token', shell=True, text=True).strip()
print(subprocess.check_output(
    f'curl -s -X DELETE -H "Authorization: Bearer {token}" '
    f'"https://{LOCATION}-aiplatform.googleapis.com/v1beta1/{REASONING_ENGINE}?force=true"',
    shell=True, text=True,
))

In [ ]:
# Optional: remove the now-stale IAM binding
run(
    f'gcloud storage buckets remove-iam-policy-binding gs://{POLICY_BUCKET} '
    f'--member="{AGENT_PRINCIPAL}" '
    f'--role="roles/storage.objectViewer" '
    f'--project={PROJECT_ID}'
)

## Troubleshooting

| Error you see | Fix |
|---|---|
| `403 storage.buckets.get` during deploy | ADC identity ≠ gcloud identity. Re-run `gcloud auth application-default login --account=<right-account>` and restart the kernel. |
| `400 Environment variable name 'GOOGLE_CLOUD_PROJECT' is reserved` | Don't put `GOOGLE_CLOUD_PROJECT` / `GOOGLE_CLOUD_LOCATION` in `env_vars` — Agent Engine injects them. |
| `ModuleNotFoundError: No module named 'policy_agent'` in engine logs | `extra_packages=['../policy_agent']` must be set so the local source ships to the container. |
| `403 storage.objects.list denied` at query time | Step 7 didn't run or was run against the wrong bucket. Re-check `POLICY_BUCKET` and the agent principal. |
| Deploy succeeds but engine won't start | `gcloud logging read 'resource.type="aiplatform.googleapis.com/ReasoningEngine" AND resource.labels.reasoning_engine_id="<ID>"' --project=<PROJECT> --limit=50 --freshness=30m` |